## 2. Ingest_clubs

Lee `clubs.json` (JSON array) desde raw y escribe `football_dev.bronze.clubs`.


In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import current_timestamp, col


In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdata1702")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/clubs.json" 


In [0]:
clubs_schema = StructType(fields=[
    StructField("club_id", IntegerType(), False),
    StructField("club_ref", StringType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("founded_year", IntegerType(), True),
    StructField("league_id", IntegerType(), True)
])


In [0]:
df_clubs = spark.read \
    .schema(clubs_schema) \
    .option("multiLine", True) \
    .json(ruta)


In [0]:
clubs_final_df = df_clubs.select(
    col("club_id"),
    col("club_ref"),
    col("name"),
    col("country"),
    col("city"),
    col("founded_year"),
    col("league_id")
).withColumn("ingestion_date", current_timestamp())


In [0]:
clubs_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.clubs")
